# PyTorchで実験を記録し、ArtifactとSweepsを使う

第2章 Modelsのサンプルです。上から順に実行してください。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nejumi/wandb_fun_book_public/blob/main/notebooks/experiment_tracking_pytorch.ipynb)

保存先は接続設定の `ENTITY`（team名）と `PROJECT` で指定できます。APIキーは `wandb.login()` の画面から入力してください。

<img src="http://wandb.me/logo-im-png" width="400" alt="Weights & Biases" />

<!--- @wandbcode{pytorch-video} -->

# Introduction
[Weights & Biases](https://wandb.com)はML 実験やデータセットのバージョン管理、リッチな可視化、レポート機能、ハイパーパレメータ探索機能、一度設定したGPU環境への簡単な計算実行、モデルのバージョン管理を可能にします。

このnotebookは、Weights & Biases（以下W&B）とPytorchのintegrationの簡単なチュートリアルです。Weights & BiasesにはCI/CDやML workflowを自動化する機能を提供していますが、**このチュートリアルではW&Bのログインから始まり、実験やデータセットのバージョン管理、リッチな可視化、レポート機能、ハイパーパレメータ探索機能などの基本的な機能を抑えます**。


それでは、W&Bをinstallするところからスタートしましょう！なお、コードの中の🐝が記されている箇所がW＆B特有のコードになります

# 🚀 W&BのInstall, Import, Log In




## W&Bのインストールとtorchの初期設定


In [ ]:
%pip install -q "wandb>=0.31,<0.32" "torch>=2.6,<3" "torchvision>=0.21,<1" numpy pandas tqdm onnx onnxscript

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset,DataLoader

from tqdm.auto import tqdm

# deterministic behaviorを設定
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# デバイス設定
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device: ",device)

# MNIST mirrorのslow mirrorsを削除
torchvision.datasets.MNIST.mirrors = [mirror for mirror in torchvision.datasets.MNIST.mirrors
                                      if not mirror.startswith("http://yann.lecun.com")]


## W&Bのインポートとログイン
まず、W&Bの[ホームページ](https://www.wandb.jp/)でアカウントを作成します（すでにアカウントをお持ちの方はそちらをご利用ください）。


早速Pytorchのコードに移ります。W&Bを利用するためには、wandbをインストール・インポートし、ログインするだけです。初期のログイン時にはAPI keyを求められますが、W&Bの画面右上の人アイコンをクリックし、User settingを押してスクロールダウンするとAPI keysが見つかりますので、それを貼り付けてください。



※なお、環境変数に登録をしておくと、毎回API keyを入力せずにログインすることが可能です。



また、W&Bの実験は大きく、エンティティ => プロジェクト => Run（実行）という単位で管理されます。エンティティはチームの単位です。デフォルトでは個人専用のエンティティになっていますが、チームのエンティティを作り、チームで同じプロジェクトを管理することが可能です。なお、個人やアカデミック利用の場合は個人のエンティティ以外にもう1つのエンティティに参加することしかできません。エンティティの下の単位が、プロジェクトです。名前の通り、MLやDLの1つのプロジェクトとしてお使いください。そのプロジェクトの中で多くの実験をする必要がありますが、プロジェクトの下でRunが管理されます。なお、エンティティとプロジェクトは手動で作成し、Runは自動的に毎回の実行で作成されていきます。下記のコードでは、今回のチュートリアルで使用するプロジェクトの名前をここで定義をしています。





In [ ]:
#!export WANDB_BASE_URL=https://your-company-here.wandb.io
#!export WANDB_API_KEY=XXX

In [ ]:
#!wandb login --relogin --host https://your-company-here.wandb.io your-api-key-here

In [ ]:
import os
import wandb

ENTITY = os.environ.get("WANDB_ENTITY", "")  # 保存先のteam名。空欄なら既定値を使用
PROJECT = os.environ.get("WANDB_PROJECT", "book-v4-mnist-pytorch")
if ENTITY:
    os.environ["WANDB_ENTITY"] = ENTITY
wandb.login()


In [ ]:
# import wandb
# wandb.login()

In [ ]:
project_name = PROJECT


# 💾 **データの準備**

## wandb連携とArtifactを用いたデータの管理


まず最初に、下記のステップでデータを準備していきます。
1.  `load_and_log`: mnistのデータを取り込み、データをtrain/validation/testに分ける
    *   下記の過程自体は標準的なPytorchの流れですが、標準的なパイプラインとの唯一の違いは、すべてを wandb.initのコンテキスト内で実行するというプロセスが入っていることです（with句を使用する以外に、すべてのコードの最後にwandb.finish()を使ってデータをアップロードする方法もありますが、with句を使用すると、エラーが生じた際なども忘れずにwandbのプロセスが終了し、必要なログがwandbに連携されるので、ベストプラクティスとしてwith句を使用することをオススメします）。
    *   データのバージョン管理にはArtifactが便利です。モデルの学習や推論のプロセスを進めるにあたり、どのデータを用いたのかをArtifactを使ってwandbに連携しておくと、実験プロセスが整理され、再現性も高くなります。

2.  `preprocess_and_log`: 正規化などの前処理を行う
    *   元データ('mnist-raw:latest')をArtifactにアップロードをし、データの処理を行う前に、run.use_artifact('mnist-raw:latest')を用いてAtifact上のどのデータを処理するのかを指定します。その過程を踏んで、処理後のデータをまたArtifactに保存すると、処理後のデータがどのデータから派生したデータであるのかの関係性(linieage)をW&B上で確認することができます。





In [ ]:
def load_and_log():
  #🐝　wandbのrunを作成
  with wandb.init(project=project_name, job_type="load-data") as run:
      datasets = load()  # separate code for loading the datasets
      names = ["training", "validation", "test"]

      #🐝 Artifactを定義
      raw_data = wandb.Artifact(
          "mnist-raw", type="dataset",
          description="Raw MNIST dataset, split into train/val/test",
          metadata={"source": "torchvision.datasets.MNIST","sizes": [len(dataset) for dataset in datasets]}
          )

      for name, subset in zip(names, datasets):
          # Artifactへの保存のためにsubdatasetをTensorDatasetに変換
          data = transform_subset2tensordataset(subset)
          with raw_data.new_file(name + ".pt", mode="wb") as file:
              x, y = data.tensors
              torch.save((x, y), file)

      #🐝 ArtifactをW&Bに保存
      run.log_artifact(raw_data)

def preprocess_and_log(steps):
  #🐝　wandbのinitialization
  with wandb.init(project=project_name, job_type="preprocess-data") as run:
      processed_data = wandb.Artifact(
          "mnist-preprocess", type="dataset",
          description="Preprocessed MNIST dataset",
          metadata=steps)

      #🐝 どのArtifactを使用するかを定義し、必要に応じてダウンロード
      raw_data_artifact = run.use_artifact('mnist-raw:v0')
      raw_dataset = raw_data_artifact.download()

      for split in ["training", "validation", "test"]:
          raw_split = read(raw_dataset, split)
          processed_dataset = preprocess(raw_split, **steps)
          with processed_data.new_file(split + ".pt", mode="wb") as file:
              x, y = processed_dataset.tensors
              torch.save((x, y), file)

      run.log_artifact(processed_data)

In [ ]:
def preprocess(dataset, normalize=True, expand_dims=True):
  x, y = dataset.tensors
  if normalize:
      # イメージを[0, 1]のrangeでスケール化
      x = x.type(torch.float32) / 255
  if expand_dims:
      # イメージのshapeを(1, 28, 28)にする
      x = torch.unsqueeze(x, 1)
  return TensorDataset(x, y)

def read(data_dir, split):
  filename = split + ".pt"
  x, y = torch.load(os.path.join(data_dir, filename))
  return TensorDataset(x, y)

def load(train_size=50000,slice=50):
  # データをtrain, testに分割
  train = torchvision.datasets.MNIST("./", train=True, download=True)
  test = torchvision.datasets.MNIST("./", train=False, download=True)
  (x_train, y_train), (x_test, y_test) = (train.data, train.targets), (test.data, test.targets)
  # trainをtrainとvalに分割
  x_train, x_val = x_train[:train_size], x_train[train_size:]
  y_train, y_val = y_train[:train_size], y_train[train_size:]
  # hands on用にデータの部分抽出
  training_set = torch.utils.data.Subset(TensorDataset(x_train, y_train), indices=range(0, len(y_train), slice))
  validation_set = torch.utils.data.Subset(TensorDataset(x_val, y_val), indices=range(0, len(y_val), slice))
  test_set = torch.utils.data.Subset(TensorDataset(x_test, y_test), indices=range(0, len(y_test), slice))

  datasets = [training_set, validation_set, test_set]
  return datasets

def transform_subset2tensordataset(subset):
  # Artifactへの保存のためにsubdatasetをTensorDatasetに変換
  subset = [subset[i] for i in range(len(subset))]
  images = torch.stack([item[0] for item in subset])
  labels = torch.tensor([item[1] for item in subset])
  data = TensorDataset(images, labels)
  return data

In [ ]:
load_and_log()
steps = {"normalize": True, "expand_dims": True}
preprocess_and_log(steps)

## Tableを用いたEDA

W&BのTableを使うと今回使用するデータをrichな可視化機能を使ってEDAをすることが可能です。ここでは、今回使用するデータをW&Bのテーブル形式に格納し、W&B上でEDAしていきます。

In [ ]:
#🐝 wandbのrunを作成
with wandb.init(project=project_name, job_type="EDA") as run:
  #  wandbのTableを作成
  table = wandb.Table(columns=["image", "label","split"])

  #🐝 どのArtifactを使用するかを定義し、必要に応じてダウンロード
  raw_data_artifact = run.use_artifact('mnist-raw:latest')
  raw_dataset = raw_data_artifact.download()
  for split in ["training", "validation", "test"]:
    raw_split = read(raw_dataset, split) #<= TensorDataset
    data_loader = DataLoader(raw_split, batch_size=100, shuffle=False)
    for _, (imgs, labels) in enumerate(tqdm(data_loader,leave=False)):
      for img, label in zip(imgs.to("cpu"), labels.to("cpu")):
        w_img = wandb.Image(img.numpy())
        table.add_data(w_img,label,split)

  run.log({'EDA_table_before_training':table})

# 👟モデルの学習

次にモデルのトレーニングを進めていきます。まずは、全体のパイプライン`model_pipeline`を定義しましょう。標準的なパイプラインとの唯一の違いは、すべてを wandb.init のコンテキスト内で実行するということです。”データの準備”の時と同じように、この関数を呼び出すことで、コードと私たちのサーバーとの間に通信路が確立されます。
先ほどとの違いは、configを定義しているところです。一般的なワークフローでは、`config` ディクショナリーにハイパーパラメータやメタデータを保存しておいて、必要に応じてそれらにアクセスします。`model_pipeline`　の中では以下のようなことを行なっています。

1. `make`: モデルアーキテクチャの定義・データローダー・損失関数・最適化手法の設定を行います。基本的には通常のPytorchのプロセスと同じです。
    * 先ほど保存したArtifactを使用することで、今回のモデルのトレーニングがどのデータを使用したかの記録をLineageに保存することができます。
2. `train`: モデルをトレーニングし、検証データに対して検証を行います。
    * ここでは wandb の`watch` と `log`の2つの関数が登場します。
        * `wandb.watch`は、Pytorchとのインテグレーションを行う際のみに使用できる関数です。モデルの勾配とパラメータを、トレーニングの log_freq ステップごとにログに記録します。トレーニングを開始する前に、単純に `wandb.watch` を呼び出すだけです。トレーニングの残りのコードは変わりません。
        * `wandb.log`はメトリクスなどの任意の情報を記録することができます。また、トレーニングのどの step であるかをオプションでログに記録することもできます。




## パイプラインの作成とwandb.initを用いたメタデータ・ハイパーパラメータの管理


In [ ]:
def model_pipeline(config):
    #🐝　wandbのrunを作成
    with wandb.init(project=project_name, config=config) as run:
      config = wandb.config
      # モデルアーキテクチャの定義・データローダー・損失関数・最適化手法の設定
      model, train_loader, validation_loader,_, criterion, optimizer = make(config, run)
      # モデルをトレーニングし、検証データに対して検証
      train(model, train_loader,validation_loader, criterion, optimizer, config, run)
      #🐝　Artifactにモデルを保存
      artifact = wandb.Artifact('model', type='model')
      with  artifact.new_file('model.pt',mode='wb') as file:
          torch.save(model.state_dict(), file)
      run.log_artifact(artifact)

In [ ]:
def make(config, run):
    # データを作成
    #🐝 どのArtifactを使用するかを定義し、必要に応じてダウンロード
    raw_data_artifact = run.use_artifact('mnist-preprocess:v0', type='dataset')
    raw_dataset = raw_data_artifact.download()
    # データローダーの作成
    train_loader = DataLoader(read(raw_dataset, "training"), config.batch_size,
                                shuffle=False, pin_memory=(device.type == "cuda"), num_workers=0)
    validation_loader = DataLoader(read(raw_dataset, "validation"), config.batch_size,
                                shuffle=False, pin_memory=(device.type == "cuda"), num_workers=0)
    test_loader = DataLoader(read(raw_dataset, "test"), config.batch_size,
                                shuffle=False, pin_memory=(device.type == "cuda"), num_workers=0)
    # モデルアーキテクチャの構築
    model = ConvNet(config.kernels,  config.dropout, config.classes).to(device)
    # 損失関数の定義
    criterion = nn.CrossEntropyLoss()
    # 最適化手法の設定
    if config.optimizer == "sgd":
        optimizer = torch.optim.SGD(model.parameters(),lr=config.learning_rate, momentum=0.9)
    elif config.optimizer == "adam":
        optimizer = torch.optim.Adam(model.parameters(),lr=config.learning_rate)

    return model, train_loader, validation_loader,test_loader, criterion, optimizer

# Conventional and convolutional neural network
class ConvNet(nn.Module):
    def __init__(self, kernels, dropout, classes=10):
        super(ConvNet, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, kernels[0], kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(kernels[0], kernels[1], kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.fc = nn.Linear(7 * 7 * kernels[-1], classes)
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        return out

In [ ]:
def train(model, training_loader, validation_loader, criterion, optimizer, config, run):
    #🐝　wandb.watchを用いてモデルのgradientsやweightsなどをトラック
    wandb.watch(model, criterion, log="all", log_freq=10)
    # 学習の開始
    table = wandb.Table(columns=["image", "pred", "target"]+[f"score_{i}" for i in range(10)], log_mode="INCREMENTAL") # wandbのTableを作成
    example_ct = 0
    for epoch in tqdm(range(config.epochs)):
        model.train()
        for step, (images, labels) in enumerate(training_loader):
            loss = train_batch(images, labels, model, optimizer, criterion)
            example_ct +=  len(images)
            #🐝　run.logを用いてモデルのlossをトラック
            run.log({"train_loss": loss}, step=example_ct, commit=False)
        validation(model, validation_loader, criterion,  run, table, log_images=config.log_prediction_iamge)

def train_batch(images, labels, model, optimizer, criterion):
    images, labels = images.to(device), labels.to(device)

    # フォワードパス ➡
    outputs = model(images)
    loss = criterion(outputs, labels)
    # バックワードパス ⬅
    optimizer.zero_grad()
    loss.backward()
    # 最適化のステップ
    optimizer.step()

    return loss

def validation(model, validation_loader, criterion, run, table,  log_images):
    model.eval()
    with torch.no_grad():
        correct, total, val_loss = 0, 0, 0
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            val_loss += criterion(outputs, labels)*labels.size(0)
            if log_images:
                log_image_table(images, predicted, labels, outputs.softmax(dim=1), run, table)
        run.log({"validation_loss":val_loss / total ,"validation_accuracy": correct / total})

def log_image_table(images, predicted, labels, probs, run, table):
    #🐝 wandbのTableを作成し、画像、予測値、実測値を保存
    for img, pred, targ, prob in zip(images.to("cpu"), predicted.to("cpu"), labels.to("cpu"), probs.to("cpu")):
        table.add_data(wandb.Image(img[0].numpy()*255), pred, targ, *prob.numpy())
    run.log({"validation_prediction_table":table}, commit=False)

### モデルパイプラインの実行

全体のパイプラインを定義し、W&Bのいくつかのコードを追加したので、トラッキングされた実験を実行する準備が整いました。
W&Bのトラッキング機能を使ってコードを実行すると、最終的に以下のリンクが出力されます：
*   ドキュメント
*   プロジェクトページ（プロジェクト内のすべての実行を整理するページ
*  実行の結果が保存されるページ

実行ページに移動し、トラックされた結果を確認してください。


In [ ]:
### 復習
"""
default_config ={'key':value}

run = wandb.init(entity=my_entity, project='aaa', config = default_config) # トラッキング開始

config = run.config # configを一旦wandbを経由して取り出す

model = create_model(aaa = config.aaa) # config情報を実験で使っていく

run.log({"key":value}) # ロギング

artifact = wandb.Artifact('aaa', type='aaa')
with  artifact.new_file('aaa.pt',mode='wb') as file:
    torch.save(model.state_dict(), file)
run.log_artifact(artifact) # アーティファクトの保存

run.finish()# トラッキング終了
"""

In [ ]:
config = dict(
    epochs=10,
    kernels=[8, 16],
    batch_size=128,
    dropout =0.1,
    learning_rate=0.005,
    optimizer="adam",
    log_prediction_iamge=True,
    classes=10,
    dataset="MNIST",
    architecture="CNN"
    )
model_pipeline(config)

In [ ]:
config = dict(
    epochs=10,
    kernels=[16, 32],
    batch_size=64,
    dropout =0.1,
    learning_rate=0.01,
    optimizer="adam",
    log_prediction_iamge=True,
    classes=10,
    dataset="MNIST",
    architecture="CNN"
    )
model_pipeline(config)

# 🏃‍♀️ハイパーパラメータチューニング

Sweepは、ハイパーパラメータおよびモデルの最適化を行うツールです。数行のコードで、ハイパーパラーメタチューニングを行うことができます。まず、最適化するハイパーパラメータの要件を定義しましょう。また、ハイパーパラメータの最適化方式（ランダム検索またはグリッド検索）を選択してから、最適化するメトリックを選択する必要があります。





## Sweepを用いたハイパーパラメータチューニング

In [ ]:
sweep_config = {
    'method': 'random',
    'metric': {
        'name': 'validation_loss',
        'goal': 'minimize'
        },
    }
parameters_dict = {
    'optimizer': {
        'values': ['adam', 'sgd']
        },
    'dropout': {
          'values': [0.1, 0.2, 0.3]
        },
    'learning_rate': {
        # 0から0.1の間を一様分布で乱数を発生
        'distribution': 'uniform',
        'min': 0,
        'max': 0.1
      },
    'batch_size': {
        # 32から256の間で対数スケールにおいて等間隔の分布から乱数を発生
        'distribution': 'q_log_uniform_values',
        'q': 8,
        'min': 32,
        'max': 256,
      },
    'kernels':{
        'values':[[16,32],[32,64]]
      },
    'epochs':{
        'value':10
    },
    'classes':{
        'value':10,
    },
    'dataset':{
        'value':'MNIST',
    },
    'architecture':{
        'value':'CNN',
    },
    'log_prediction_iamge':{
        'value':False
    },
    }

sweep_config['parameters'] = parameters_dict

#🐝 Sweep用に関数を作成
def model_pipeline_sweep(config=None):
    #🐝　wandbのrunを作成
    with wandb.init(config=config) as run:
      config = run.config
      # モデルアーキテクチャの定義・データローダー・損失関数・最適化手法の設定
      model, train_loader, validation_loader, _,criterion, optimizer = make(config, run)
      # モデルをトレーニングし、検証データに対して検証
      train(model, train_loader, validation_loader, criterion, optimizer, config, run)
      #🐝　Artifactにモデルを保存
      artifact = wandb.Artifact('model', type='model')
      with  artifact.new_file('model.pt',mode='wb') as file:
          torch.save(model.state_dict(), file)
      run.log_artifact(artifact)

In [ ]:
sweep_id = wandb.sweep(sweep_config, project=project_name)
wandb.agent(sweep_id=sweep_id, function=model_pipeline_sweep, count=10)

# 🧪テストデータに対する精度評価とArtifactとModelを用いた最終モデルの管理

## 最終モデルの獲得と検証

In [ ]:
wandb.teardown()


In [ ]:
# Sweepの評価指標に従って最良のRunを取得します。
api = wandb.Api()
sweep_entity = ENTITY or api.default_entity
best_run = api.sweep(f"{sweep_entity}/{project_name}/{sweep_id}").best_run()
if best_run is None:
    raise RuntimeError("Sweepに評価済みのRunがありません。上のagent実行セルを確認してください。")
best_sweep_path = "/".join(best_run.path)
best_sweep_path


In [ ]:
# username, project, run_idを指定して、もっとも精度の良いモデルのconfigを取得し
final_config = wandb.Api().run(best_sweep_path).config

#sweepでは予測画像の保存を行っていないため、その設定だけ変更します
final_config.update({'log_prediction_iamge':True})

## 最終モデルの検証

In [ ]:
def test_pipeline(config):
    #🐝　wandbのrunを作成
    with wandb.init(project=project_name, config=config) as run:
      config = wandb.config
      # モデルアーキテクチャの定義・データローダー・損失関数・最適化手法の設定
      model, train_loader, validation_loader,test_loader, criterion, optimizer = make(config, run)
      # モデルをトレーニングし、検証データに対して検証
      train(model, train_loader,validation_loader, criterion, optimizer, config, run)
      # テストデータで検証
      test(model, test_loader, criterion, run, config)

      #🐝　Artifactにモデルを保存
      artifact = wandb.Artifact('model', type='model')
      with  artifact.new_file('model.pt',mode='wb') as file:
          torch.save(model.state_dict(), file)
      run.log_artifact(artifact)

def test(model, test_loader, criterion, run, config):
    model.eval()
    table = wandb.Table(columns=["image", "pred", "target"]+[f"score_{i}" for i in range(10)]) # wandbのTableを作成
    with torch.no_grad():
        correct, total, test_loss = 0, 0, 0
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            test_loss += criterion(outputs, labels)*labels.size(0)
            if config.log_prediction_iamge:
                for img, pred, targ, prob in zip(images.to("cpu"), predicted.to("cpu"), labels.to("cpu"), outputs.softmax(dim=1).to("cpu")):
                  table.add_data(wandb.Image(img[0].numpy()*255), pred, targ, *prob.numpy())
                run.log({"test(holdout)_prediction_table":table}, commit=False)
        run.log({"test_loss(holdout)":test_loss / total ,"test_accuracy(holdout)": correct / total})

In [ ]:
test_pipeline(final_config)